# EN3150 Assignment 03: Resource-Constrained Image Classification
 
**GitHub profile:** [https://github.com/Achintha-Bandara/waste-classifier](https://github.com/Achintha-Bandara/waste-classifier)  
**Dataset:** [UCI RealWaste](https://archive.ics.uci.edu/dataset/908/realwaste)

This report studies two custom convolutional neural networks for nine-class waste-image classification at a resource-constrained input resolution. Model A is a conventional CNN, while Model B uses depthwise-separable convolutions and is constrained to fewer than 100,000 trainable parameters. MobileNetV2 and EfficientNet-B0 are also fine-tuned as pretrained lightweight references.

The notebook is executable from beginning to end. Run the data-preparation cell, the architecture cells, each optimizer experiment, and the pretrained-model cells to generate the curves, confusion matrices, metrics, and comparisons shown in this report. The interpretation below is intentionally tied to the results generated in the current notebook run; therefore, the best optimizer or model may change when the experiments are rerun.


## 1. Data preparation

The selected dataset is **RealWaste**, an image dataset with nine classes: Cardboard, Food Organics, Glass, Metal, Miscellaneous Trash, Paper, Plastic, Textile Trash, and Vegetation. The data-preparation cell below discovers class folders, creates a deterministic 70%/15%/15% stratified split with seed 42, resizes every RGB image to **64 x 64**, and scales pixels to `[0, 1]`. The same split is reused by both custom CNNs and the pretrained networks.

The assignment excludes CIFAR-10; RealWaste is an appropriate real-world low-resolution classification problem. Run this cell before any training cell.


### Data loader and result visualisation

The shared data loader prepares the dataset and the visualisation utilities present histories, metrics, predictions, and figures alongside each experiment.


In [5]:
# Training setup: imports, deterministic split, and result visualisation utilities.
import time
from pathlib import Path
from collections import Counter
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

ROOT = Path.cwd()
if not (ROOT / "dataset6").exists():
    ROOT = Path("/mnt/c/Users/achin/Documents/Semester 5 Modules/EN3150 - Pattern Recognition/waste-classifier")
DATASET_DIR = ROOT / "dataset6"
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 50
SEED = 42
AUG_MODE = "none"
USE_CACHE = True

class_names = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
class_to_idx = {name: i for i, name in enumerate(class_names)}
paths, labels = [], []
for class_name in class_names:
    for path in sorted((DATASET_DIR / class_name).rglob("*")):
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            paths.append(str(path)); labels.append(class_to_idx[class_name])
X_train, X_temp, y_train, y_temp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f"{len(paths)} images | train={len(X_train)} | validation={len(X_val)} | test={len(X_test)}")
print(f"Classes: {class_names}")

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    return tf.cast(image, tf.float32) / 255.0, label

def make_dataset(paths_, labels_, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths_, labels_)).map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if USE_CACHE: ds = ds.cache()
    if training: ds = ds.shuffle(len(paths_), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

OPTIMIZERS = {
    "adam": lambda: keras.optimizers.Adam(1e-4),
    "adam_1e-3": lambda: keras.optimizers.Adam(1e-3),
    "sgd": lambda: keras.optimizers.SGD(1e-2, momentum=0.0),
    "sgd_m0.5": lambda: keras.optimizers.SGD(1e-2, momentum=0.5),
    "sgd_m0.9": lambda: keras.optimizers.SGD(1e-2, momentum=0.9),
}

def inline_result(model, history, title, train_seconds):
    loss, accuracy = model.evaluate(test_ds, verbose=0)
    logits = model.predict(test_ds, verbose=0)
    pred = logits.argmax(axis=1)
    report = classification_report(y_test, pred, target_names=class_names, zero_division=0)
    print(f"{title}: test accuracy={accuracy:.4f}, macro F1={f1_score(y_test, pred, average='macro', zero_division=0):.4f}, time={train_seconds:.1f}s")
    print(report)
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    axes[0].plot(history.history['loss'], label='train'); axes[0].plot(history.history['val_loss'], label='validation'); axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].plot(history.history['accuracy'], label='train'); axes[1].plot(history.history['val_accuracy'], label='validation'); axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=.25)
    axes[2].imshow(confusion_matrix(y_test, pred), cmap='Blues'); axes[2].set_title('Confusion matrix'); axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
    plt.tight_layout(); plt.show()
    return {'model': title, 'test_loss': float(loss), 'test_accuracy': float(accuracy), 'test_precision_macro': float(precision_score(y_test, pred, average='macro', zero_division=0)), 'test_recall_macro': float(recall_score(y_test, pred, average='macro', zero_division=0)), 'test_f1_macro': float(f1_score(y_test, pred, average='macro', zero_division=0)), 'parameters': model.count_params(), 'history': history.history, 'confusion_matrix': confusion_matrix(y_test, pred)}

class InMemoryBestWeights(keras.callbacks.Callback):
    """Keep the lowest-validation-loss weights without writing a checkpoint file."""
    def __init__(self):
        super().__init__()
        self.best = np.inf
        self.weights = None
    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get('val_loss')
        if value is not None and value < self.best:
            self.best = float(value)
            self.weights = self.model.get_weights()

NOTEBOOK_RESULTS = {}


I0000 00:00:1792398873.802896    5876 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792398876.222672    5876 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792398881.941905    5876 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


4752 images | train=3326 | validation=713 | test=713
Classes: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


I0000 00:00:1792398905.506336    5876 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1341 MB memory:  -> device: 0, name: NVIDIA GeForce MX550, pci bus id: 0000:01:00.0, compute capability: 7.5


## 2. Custom CNN architecture design

### Model A: standard CNN

Model A uses standard `3 x 3` convolutions in three VGG-style stages. Each stage has two convolution, batch-normalization, and ReLU sequences followed by `2 x 2` max pooling. The filter counts are 32, 64, and 128. The final `8 x 8 x 128` tensor is flattened into a 128-unit dense layer and a nine-unit logits output.

### Model B: lightweight CNN

Model B begins with a standard 3-to-16 convolution and then uses depthwise-separable convolution blocks with 32, 64, 64, 128, 128, and 256 filters. Four max-pooling layers reduce the spatial resolution, global average pooling avoids a large flattening layer, and the head contains dense layers of 64 and 9 units. ReLU/ReLU6 are piecewise-linear and hardware-friendly; the final logits are passed to a numerically stable cross-entropy loss rather than applying softmax at every intermediate layer.

The architecture cells construct the models used by the experiments and report their parameter counts. Model B is checked against the assignment limit of 100,000 trainable parameters.


## Model A — standard CNN training

In [ ]:
def build_modelA():
    def stage(filters, dropout=0.0):
        block = []
        for _ in range(2):
            block += [layers.Conv2D(filters, (3, 3), padding="same", use_bias=False), layers.BatchNormalization(), layers.Activation("relu")]
        block.append(layers.MaxPooling2D((2, 2)))
        if dropout: block.append(layers.Dropout(dropout))
        return block
    return keras.Sequential([layers.Input((64, 64, 3))] + stage(32) + stage(64, .10) + stage(128, .10) + [layers.Flatten(), layers.Dense(128, activation='relu'), layers.Dropout(.25), layers.Dense(len(class_names))], name='ModelA_Standard_CNN_v2')

def train_modelA(optimizer_key, epochs=EPOCHS):
    keras.utils.set_random_seed(SEED)
    model = build_modelA()
    model.compile(optimizer=OPTIMIZERS[optimizer_key](), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    best = InMemoryBestWeights()
    start = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[best, keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=.5, patience=4, min_lr=1e-6)], verbose=2)
    model.set_weights(best.weights)
    result = inline_result(model, history, f'Model A / {optimizer_key}', time.perf_counter()-start)
    NOTEBOOK_RESULTS[f'ModelA_{optimizer_key}'] = result
    return result


## 3. Optimizer selection and tuning

The experiments cover Adam (`1e-4`), Adam (`1e-3`), vanilla SGD (`0.01`), SGD with momentum 0.5, and SGD with momentum 0.9. Each optimizer is trained separately for both custom models. Every experiment trains for 50 epochs by default, restores the weights with the lowest validation loss, and presents its test metrics, classification report, loss/accuracy curves, and confusion matrix below the experiment.

Momentum accumulates a fraction of the previous update, reducing oscillation and accelerating movement along consistent gradient directions. The optimizer comparison cells provide the empirical basis for selecting the final optimizer for each model. Since the experiments are executable, conclusions should be based on the metrics produced in the current run rather than on a fixed prior ranking.


### `python modelA.py --optimizer=adam`

In [ ]:
MODEL_A_adam  = train_modelA('adam')


### `python modelA.py --optimizer=adam_1e-3`

In [ ]:
MODEL_A_adam_1e_3  = train_modelA('adam_1e-3')


### `python modelA.py --optimizer=sgd`

In [ ]:
MODEL_A_sgd  = train_modelA('sgd')


### `python modelA.py --optimizer=sgd_m0.5`

In [ ]:
MODEL_A_sgd_m0_5  = train_modelA('sgd_m0.5')


### `python modelA.py --optimizer=sgd_m0.9`

In [ ]:
MODEL_A_sgd_m0_9  = train_modelA('sgd_m0.9')


In [ ]:
if False:
    """Train modelA with one selected optimizer."""
    import os
    import sys
    import json
    import time
    import argparse
    from pathlib import Path
    from datetime import datetime, timezone
    from collections import Counter
    
    import numpy as np
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import matplotlib.pyplot as plt
    
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        ConfusionMatrixDisplay,
        f1_score,
        precision_score,
        recall_score,
    )
    
    OPTIMIZER_REGISTRY = {
        "adam": {"label": "Adam (lr=1e-4)",
                 "make": lambda: keras.optimizers.Adam(learning_rate=1e-4)},
        "adam_1e-3": {"label": "Adam (lr=1e-3)",
                      "make": lambda: keras.optimizers.Adam(learning_rate=1e-3)},
        "sgd": {"label": "SGD (lr=0.01, momentum=0)",
                "make": lambda: keras.optimizers.SGD(learning_rate=1e-2, momentum=0.0)},
        "sgd_m0.5": {"label": "SGD + Momentum 0.5 (lr=0.01)",
                     "make": lambda: keras.optimizers.SGD(learning_rate=1e-2, momentum=0.5)},
        "sgd_m0.9": {"label": "SGD + Momentum 0.9 (lr=0.01)",
                     "make": lambda: keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9)},
    }
    parser = argparse.ArgumentParser(
        description="Train modelA (standard CNN) with one optimizer.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument("--arch", choices=["base", "deep"],
                        default=os.environ.get("ARCH", "deep"),
                        help="base = original topology, deep = 2 convs per stage")
    parser.add_argument("--aug", choices=["none", "geometric", "full"],
                        default=os.environ.get("AUG_MODE", "none"),
                        help="training-only augmentation")
    parser.add_argument("--epochs", type=int,
                        default=int(os.environ.get("EPOCHS", 50)),
                        help="epochs per optimizer (assignment requires >= 20)")
    parser.add_argument("--optimizer", required=True, choices=list(OPTIMIZER_REGISTRY),
                        help="optimizer key")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--dataset", default=os.environ.get("DATASET_DIR"),
                        help="dataset root (one sub-folder per class); "
                             "default: <script dir>/dataset6")
    parser.add_argument("--no-cache", action="store_true",
                        help="do not cache decoded images in RAM")
    args = parser.parse_args()
    
    if args.epochs < 20:
        print(f"WARNING: --epochs={args.epochs} is below the assignment minimum of 20.")
    
    OPTIMIZER_KEY = args.optimizer
    OPTIMIZER = {"key": OPTIMIZER_KEY, **OPTIMIZER_REGISTRY[OPTIMIZER_KEY]}
    MODEL_LABEL = "modelA"
    PARAM_LIMIT = None
    
    PROJECT_DIR = Path(__file__).resolve().parent
    DATASET_DIR = Path(args.dataset) if args.dataset else PROJECT_DIR / "dataset6"
    IMG_SIZE = (64, 64)
    BATCH_SIZE = args.batch_size
    SEED = args.seed
    EPOCHS = args.epochs
    AUG_MODE = args.aug
    ARCH = args.arch
    USE_CACHE = not args.no_cache
    RESULTS_DIR = PROJECT_DIR / "results" / "modelA" / OPTIMIZER_KEY
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    COMMAND_LINE = "python " + " ".join(
        [Path(sys.argv[0]).name] + sys.argv[1:])
    print(f"\nCommand : {COMMAND_LINE}")
    print(f"Arguments: arch={ARCH} aug={AUG_MODE} epochs={EPOCHS} "
          f"optimizer={OPTIMIZER_KEY} batch={BATCH_SIZE} seed={SEED} cache={USE_CACHE}")
    (RESULTS_DIR / "command.txt").write_text(
        COMMAND_LINE + "\n" + json.dumps(vars(args), indent=2) + "\n",
        encoding="utf-8")
    
    class_names = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths, labels = [], []
    for class_name in class_names:
        for path in sorted((DATASET_DIR / class_name).rglob("*")):
            if path.is_file() and path.suffix.lower() in extensions:
                paths.append(str(path))
                labels.append(class_to_idx[class_name])
    
    if not paths:
        raise ValueError(f"No images found in {DATASET_DIR}")
    
    class_counts = Counter(labels)
    print("\nImages per class:")
    for name, idx in class_to_idx.items():
        print(f"{name:30s}: {class_counts[idx]}")
    print(f"\nTotal images: {len(paths)}")
    print(f"Number of classes: {num_classes}")
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
    
    print(f"\nTraining:   {len(X_train)} ({len(X_train)/len(paths):.1%})")
    print(f"Validation: {len(X_val)} ({len(X_val)/len(paths):.1%})")
    print(f"Testing:    {len(X_test)} ({len(X_test)/len(paths):.1%})")
    
    
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)  # downscale to 64x64
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    
    def build_augmentation():
        """Train-only augmentation (no trainable parameters, not part of the model).
        geometric: flips + mild rotation/zoom (colour is left untouched, because
        colour is a key cue for glass / vegetation / cardboard).
        full: adds translation, contrast and brightness jitter."""
        ops = [layers.RandomFlip("horizontal_and_vertical"),
               layers.RandomRotation(0.10, fill_mode="reflect"),
               layers.RandomZoom(0.10, fill_mode="reflect")]
        if AUG_MODE == "full":
            ops += [layers.RandomTranslation(0.10, 0.10, fill_mode="reflect"),
                    layers.RandomContrast(0.20),
                    layers.RandomBrightness(0.20, value_range=(0.0, 1.0))]
        return keras.Sequential(ops, name="train_augmentation")
    
    
    augmentation = build_augmentation()
    
    
    def make_dataset(paths_, labels_, training=False):
        ds = tf.data.Dataset.from_tensor_slices((paths_, labels_))
        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        if USE_CACHE:
            ds = ds.cache()  # cache the clean decoded images; augment afterwards
        if training:
            ds = ds.shuffle(len(paths_), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE)
        if training and AUG_MODE != "none":
            ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)
    
    
    train_ds = make_dataset(X_train, y_train, training=True)
    val_ds = make_dataset(X_val, y_val)
    test_ds = make_dataset(X_test, y_test)
    
    
    def build_model_deep():
        """modelA deep: VGG-style stages, each = 2 x (Conv2D 3x3 -> BN -> ReLU) then
        MaxPooling2D. Output = logits."""
        def stage(filters, dropout=0.0):
            s = []
            for _ in range(2):
                s += [layers.Conv2D(filters, kernel_size=(3, 3), padding="same",
                                    use_bias=False),
                      layers.BatchNormalization(),
                      layers.Activation("relu")]
            s.append(layers.MaxPooling2D(pool_size=(2, 2)))
            if dropout:
                s.append(layers.Dropout(dropout))
            return s
    
        return keras.Sequential(
            [layers.Input(shape=(64, 64, 3))]
            + stage(32) + stage(64, 0.10) + stage(128, 0.10)
            + [layers.Flatten(),
               layers.Dense(128, activation="relu"),
               layers.Dropout(0.25),
               layers.Dense(num_classes)],  # logits; softmax folded into the loss
            name="ModelA_Standard_CNN_v2")
    
    
    def build_model_base():
        """modelA: standard Conv2D + MaxPool2D CNN. Output = logits."""
        return keras.Sequential([
            layers.Input(shape=(64, 64, 3)),
    
            layers.Conv2D(32, kernel_size=(3, 3), padding="same",
                          activation="relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
    
            layers.Conv2D(64, kernel_size=(3, 3), padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Dropout(0.10),
    
            layers.Conv2D(128, kernel_size=(3, 3), padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
    
            layers.Flatten(),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.25),
            layers.Dense(num_classes),  # logits; softmax folded into the loss
        ], name="ModelA_Standard_CNN")
    
    
    def build_model():
        return build_model_deep() if ARCH == "deep" else build_model_base()
    def layer_breakdown(model):
        """Per-layer output shape, parameters (total / trainable) and MACs."""
        rows = []
        for layer in model.layers:
            out_shape = tuple(layer.output.shape[1:])
            in_shape = tuple(layer.input.shape[1:])
            trainable = sum(int(np.prod(w.shape)) for w in layer.trainable_weights)
            total = layer.count_params()
            macs = 0
            if isinstance(layer, layers.SeparableConv2D):
                k = layer.kernel_size[0] * layer.kernel_size[1]
                cin, cout = in_shape[-1], layer.filters
                macs = out_shape[0] * out_shape[1] * (k * cin + cin * cout)
            elif isinstance(layer, layers.Conv2D):
                k = layer.kernel_size[0] * layer.kernel_size[1]
                cin, cout = in_shape[-1], layer.filters
                macs = out_shape[0] * out_shape[1] * k * cin * cout
            elif isinstance(layer, layers.Dense):
                macs = in_shape[-1] * out_shape[-1]
            rows.append({
                "layer": layer.name,
                "type": layer.__class__.__name__,
                "output_shape": list(out_shape),
                "total_params": int(total),
                "trainable_params": int(trainable),
                "macs": int(macs),
            })
        return rows
    
    
    def format_breakdown(rows):
        header = f"{'Layer':28s}{'Type':22s}{'Output shape':18s}{'Params':>10s}{'Trainable':>12s}{'MACs':>14s}"
        lines = [header, "-" * len(header)]
        for r in rows:
            lines.append(
                f"{r['layer']:28s}{r['type']:22s}{str(tuple(r['output_shape'])):18s}"
                f"{r['total_params']:>10,}{r['trainable_params']:>12,}{r['macs']:>14,}")
        lines.append("-" * len(header))
        lines.append(
            f"{'TOTAL':28s}{'':22s}{'':18s}"
            f"{sum(r['total_params'] for r in rows):>10,}"
            f"{sum(r['trainable_params'] for r in rows):>12,}"
            f"{sum(r['macs'] for r in rows):>14,}")
        return "\n".join(lines)
    
    
    class EpochTimer(keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.epoch_times = []
    
        def on_epoch_begin(self, epoch, logs=None):
            self.t0 = time.perf_counter()
    
        def on_epoch_end(self, epoch, logs=None):
            self.epoch_times.append(float(time.perf_counter() - self.t0))
    
    
    def first_epoch_reaching(values, threshold, mode="ge"):
        """1-based epoch at which `values` first reaches the threshold, else None."""
        for i, v in enumerate(values):
            if (mode == "ge" and v >= threshold) or (mode == "le" and v <= threshold):
                return i + 1
        return None
    
    
    def run_experiment(opt_cfg):
        """Train a fresh modelA with one optimizer for EPOCHS epochs (no early
        stopping, so every run completes >= 20 epochs), restore the best
        validation-loss weights, then evaluate on the test set."""
        print(f"\n{'=' * 70}\nTraining {MODEL_LABEL} with {opt_cfg['label']}\n{'=' * 70}")
        keras.utils.set_random_seed(SEED)  # identical initial weights per optimizer
        model = build_model()
        model.compile(
            optimizer=opt_cfg["make"](),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )
    
        weights_path = RESULTS_DIR / f"best_{opt_cfg['key']}.weights.h5"
        timer = EpochTimer()
        callbacks = [
            keras.callbacks.ModelCheckpoint(
                filepath=str(weights_path), monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6,
                mode="min", verbose=1),
            timer,
        ]
    
        t0 = time.perf_counter()
        history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
                            callbacks=callbacks, verbose=2)
        total_time = time.perf_counter() - t0
    
        if weights_path.exists():
            model.load_weights(weights_path)  # best validation-loss weights
    
        test_loss, test_acc = model.evaluate(test_ds, verbose=0)
        logits = model.predict(test_ds, verbose=0)
        preds = logits.argmax(axis=1)
    
        hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        val_acc = history.history["val_accuracy"]
        best_vl = float(min(val_loss))
        last10 = val_loss[-10:]
        # The first epoch includes XLA/graph compilation, so the median (and the
        # mean excluding epoch 1) are the fair "time per epoch" figures.
        et = timer.epoch_times
        result = {
            "key": opt_cfg["key"],
            "label": opt_cfg["label"],
            "epochs_completed": len(val_loss),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "test_f1_macro": float(f1_score(y_test, preds, average="macro", zero_division=0)),
            "test_f1_weighted": float(f1_score(y_test, preds, average="weighted", zero_division=0)),
            "test_precision_macro": float(precision_score(y_test, preds, average="macro", zero_division=0)),
            "test_recall_macro": float(recall_score(y_test, preds, average="macro", zero_division=0)),
            "test_confusion_matrix": confusion_matrix(
                y_test, preds, labels=list(range(num_classes))).tolist(),
            "best_validation_loss": best_vl,
            "best_validation_loss_epoch": int(np.argmin(val_loss)) + 1,
            "best_validation_accuracy": float(max(val_acc)),
            "best_validation_accuracy_epoch": int(np.argmax(val_acc)) + 1,
            "final_training_loss": float(train_loss[-1]),
            "final_training_accuracy": float(history.history["accuracy"][-1]),
            "final_validation_loss": float(val_loss[-1]),
            "final_validation_accuracy": float(val_acc[-1]),
            "epochs_to_val_acc_60": first_epoch_reaching(val_acc, 0.60),
            "epochs_to_val_acc_70": first_epoch_reaching(val_acc, 0.70),
            "epochs_to_val_loss_1.0": first_epoch_reaching(val_loss, 1.0, "le"),
            "epochs_to_within_5pct_best_val_loss": first_epoch_reaching(
                val_loss, best_vl * 1.05, "le"),
            "train_loss_epoch_5": float(train_loss[4]) if len(train_loss) >= 5 else None,
            "train_loss_epoch_10": float(train_loss[9]) if len(train_loss) >= 10 else None,
            "val_loss_std_last10": float(np.std(last10)),
            "generalisation_gap_acc": float(
                history.history["accuracy"][-1] - val_acc[-1]),
            "total_training_time_seconds": float(total_time),
            "average_epoch_time_seconds": float(np.mean(et)),
            "average_epoch_time_excl_first_seconds": float(np.mean(et[1:])) if len(et) > 1 else float(np.mean(et)),
            "median_epoch_time_seconds": float(np.median(et)),
            "epoch_time_seconds": et,
            "history": hist,
        }
        print(f"{opt_cfg['label']}: test acc {test_acc:.4f} | macro F1 "
              f"{result['test_f1_macro']:.4f} | best val loss "
              f"{best_vl:.4f} @ epoch {result['best_validation_loss_epoch']} | "
              f"{result['median_epoch_time_seconds']:.2f}s/epoch (median)")
        return model, preds, result, weights_path
    
    
    keras.utils.set_random_seed(SEED)
    arch_model = build_model()
    arch_model.summary()
    rows = layer_breakdown(arch_model)
    breakdown_text = format_breakdown(rows)
    print("\n" + breakdown_text)
    
    trainable_params = sum(r["trainable_params"] for r in rows)
    total_params = sum(r["total_params"] for r in rows)
    non_trainable_params = total_params - trainable_params
    total_macs = sum(r["macs"] for r in rows)
    float32_size_kb = total_params * 4 / 1024
    
    print(f"\nTrainable parameters:     {trainable_params:,}")
    print(f"Non-trainable parameters: {non_trainable_params:,}")
    print(f"Total parameters:         {total_params:,}")
    print(f"MACs per inference:       {total_macs:,}")
    print(f"Float32 weight size:      {float32_size_kb:.1f} KB")
    
    if PARAM_LIMIT is not None:
        assert trainable_params <= PARAM_LIMIT, (
            f"{MODEL_LABEL} exceeds the {PARAM_LIMIT:,} trainable-parameter "
            f"limit: {trainable_params:,}")
        print(f"Parameter constraint satisfied (<= {PARAM_LIMIT:,}).")
    
    (RESULTS_DIR / "layer_breakdown.txt").write_text(breakdown_text, encoding="utf-8")
    (RESULTS_DIR / "layer_breakdown.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")
    del arch_model
    
    main_model, main_preds, main_res, weights_path = run_experiment(OPTIMIZER)
    weights_path.unlink(missing_ok=True)
    model_path = RESULTS_DIR / "model.keras"
    main_model.save(model_path)
    disk_size_kb = model_path.stat().st_size / 1024
    print(f"\nSaved model file size: {disk_size_kb:.1f} KB")
    
    report_dict = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, output_dict=True, zero_division=0)
    report_text = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, zero_division=0)
    print("\n" + report_text)
    
    cm = confusion_matrix(y_test, main_preds, labels=list(range(num_classes)))
    
    final_metrics = {
        "command": COMMAND_LINE,
        "model": MODEL_LABEL,
        "architecture": ARCH,
        "augmentation": AUG_MODE,
        "optimizer": main_res["label"],
        "test_loss": main_res["test_loss"],
        "test_accuracy": main_res["test_accuracy"],
        "test_precision_macro": main_res["test_precision_macro"],
        "test_recall_macro": main_res["test_recall_macro"],
        "test_f1_macro": main_res["test_f1_macro"],
        "test_f1_weighted": main_res["test_f1_weighted"],
        "test_classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
        "trainable_parameters": int(trainable_params),
        "non_trainable_parameters": int(non_trainable_params),
        "total_parameters": int(total_params),
        "macs_per_inference": int(total_macs),
        "float32_weight_size_kb": float(float32_size_kb),
        "model_file_size_kb": float(disk_size_kb),
        "epochs_completed": main_res["epochs_completed"],
        "best_validation_loss": main_res["best_validation_loss"],
        "best_validation_loss_epoch": main_res["best_validation_loss_epoch"],
        "best_validation_accuracy": main_res["best_validation_accuracy"],
        "best_validation_accuracy_epoch": main_res["best_validation_accuracy_epoch"],
        "final_training_loss": main_res["final_training_loss"],
        "final_training_accuracy": main_res["final_training_accuracy"],
        "final_validation_loss": main_res["final_validation_loss"],
        "final_validation_accuracy": main_res["final_validation_accuracy"],
        "total_training_time_seconds": main_res["total_training_time_seconds"],
        "average_epoch_time_seconds": main_res["average_epoch_time_seconds"],
        "average_epoch_time_excl_first_seconds": main_res["average_epoch_time_excl_first_seconds"],
        "median_epoch_time_seconds": main_res["median_epoch_time_seconds"],
    }
    (RESULTS_DIR / "metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
    (RESULTS_DIR / "training_history.json").write_text(
        json.dumps({**main_res["history"], "epoch_time_seconds": main_res["epoch_time_seconds"]}, indent=2),
        encoding="utf-8")
    (RESULTS_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
    np.savetxt(RESULTS_DIR / "confusion_matrix.csv", cm, delimiter=",", fmt="%d")
    
    (RESULTS_DIR / "run_metadata.json").write_text(json.dumps({
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "command": COMMAND_LINE,
        "arguments": vars(args),
        "model": MODEL_LABEL,
        "dataset": str(DATASET_DIR),
        "image_size": list(IMG_SIZE),
        "batch_size": BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "seed": SEED,
        "optimizer": main_res["label"],
        "loss": "SparseCategoricalCrossentropy(from_logits=True)",
        "augmentation": AUG_MODE,
        "architecture": ARCH,
        "lr_schedule": "ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6)",
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "num_images": len(paths),
        "train_images": len(X_train),
        "validation_images": len(X_val),
        "test_images": len(X_test),
    }, indent=2), encoding="utf-8")
    
    plt.figure(figsize=(12, 6))
    plt.bar(class_names, [class_counts[class_to_idx[n]] for n in class_names])
    plt.title("Images per Class")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
    plt.close()
    
    epochs_axis = range(1, main_res["epochs_completed"] + 1)
    h = main_res["history"]
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, h["loss"], label="Training Loss")
    plt.plot(epochs_axis, h["val_loss"], label="Validation Loss")
    plt.title(f"{MODEL_LABEL}: Training and Validation Loss ({main_res['label']})")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=150)
    plt.close()
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, h["accuracy"], label="Training Accuracy")
    plt.plot(epochs_axis, h["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{MODEL_LABEL}: Training and Validation Accuracy ({main_res['label']})")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "accuracy_curve.png", dpi=150)
    plt.close()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
    ax.set_title(f"{MODEL_LABEL}: Test Confusion Matrix ({main_res['label']})")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()
    
    print(f"\nResults saved to: {RESULTS_DIR}")


## Model B — lightweight depthwise-separable CNN training

In [ ]:
def sep_block(filters):
    return [layers.SeparableConv2D(filters, (3, 3), padding='same', use_bias=False), layers.BatchNormalization(), layers.ReLU(max_value=6.0)]

def build_modelB():
    return keras.Sequential([layers.Input((64, 64, 3)), layers.Conv2D(16, (3, 3), padding='same', use_bias=False), layers.BatchNormalization(), layers.ReLU(max_value=6.0), layers.MaxPooling2D((2,2)), *sep_block(32), layers.MaxPooling2D((2,2)), *sep_block(64), *sep_block(64), layers.MaxPooling2D((2,2)), layers.Dropout(.10), *sep_block(128), *sep_block(128), layers.MaxPooling2D((2,2)), layers.Dropout(.10), *sep_block(256), layers.GlobalAveragePooling2D(), layers.Dense(64, activation='relu'), layers.Dropout(.25), layers.Dense(len(class_names))], name='ModelB_Lightweight_SeparableCNN')

def train_modelB(optimizer_key, epochs=EPOCHS):
    keras.utils.set_random_seed(SEED)
    model = build_modelB()
    model.compile(optimizer=OPTIMIZERS[optimizer_key](), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    assert model.count_params() <= 100000, model.count_params()
    best = InMemoryBestWeights()
    start = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[best, keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=.5, patience=4, min_lr=1e-6)], verbose=2)
    model.set_weights(best.weights)
    result = inline_result(model, history, f'Model B / {optimizer_key}', time.perf_counter()-start)
    NOTEBOOK_RESULTS[f'ModelB_{optimizer_key}'] = result
    return result


### `python modelB.py --optimizer=adam`

In [ ]:
MODEL_B_adam  = train_modelB('adam')


### `python modelB.py --optimizer=adam_1e-3`

In [ ]:
MODEL_B_adam_1e_3  = train_modelB('adam_1e-3')


### `python modelB.py --optimizer=sgd`

In [ ]:
MODEL_B_sgd  = train_modelB('sgd')


### `python modelB.py --optimizer=sgd_m0.5`

In [ ]:
MODEL_B_sgd_m0_5  = train_modelB('sgd_m0.5')


### `python modelB.py --optimizer=sgd_m0.9`

In [ ]:
MODEL_B_sgd_m0_9  = train_modelB('sgd_m0.9')


In [ ]:
if False:
    """Train modelB with one selected optimizer."""
    import os
    import sys
    import json
    import time
    import argparse
    from pathlib import Path
    from datetime import datetime, timezone
    from collections import Counter
    
    import numpy as np
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import matplotlib.pyplot as plt
    
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        ConfusionMatrixDisplay,
        f1_score,
        precision_score,
        recall_score,
    )
    
    OPTIMIZER_REGISTRY = {
        "adam": {"label": "Adam (lr=1e-4)",
                 "make": lambda: keras.optimizers.Adam(learning_rate=1e-4)},
        "adam_1e-3": {"label": "Adam (lr=1e-3)",
                      "make": lambda: keras.optimizers.Adam(learning_rate=1e-3)},
        "sgd": {"label": "SGD (lr=0.01, momentum=0)",
                "make": lambda: keras.optimizers.SGD(learning_rate=1e-2, momentum=0.0)},
        "sgd_m0.5": {"label": "SGD + Momentum 0.5 (lr=0.01)",
                     "make": lambda: keras.optimizers.SGD(learning_rate=1e-2, momentum=0.5)},
        "sgd_m0.9": {"label": "SGD + Momentum 0.9 (lr=0.01)",
                     "make": lambda: keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9)},
    }
    parser = argparse.ArgumentParser(
        description="Train modelB (depthwise separable CNN) with one optimizer.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument("--aug", choices=["none", "geometric", "full"],
                        default=os.environ.get("AUG_MODE", "none"),
                        help="training-only augmentation")
    parser.add_argument("--epochs", type=int,
                        default=int(os.environ.get("EPOCHS", 50)),
                        help="epochs per optimizer (assignment requires >= 20)")
    parser.add_argument("--optimizer", required=True, choices=list(OPTIMIZER_REGISTRY),
                        help="optimizer key")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--dataset", default=os.environ.get("DATASET_DIR"),
                        help="dataset root (one sub-folder per class); "
                             "default: <script dir>/dataset6")
    parser.add_argument("--no-cache", action="store_true",
                        help="do not cache decoded images in RAM")
    args = parser.parse_args()
    
    if args.epochs < 20:
        print(f"WARNING: --epochs={args.epochs} is below the assignment minimum of 20.")
    
    OPTIMIZER_KEY = args.optimizer
    OPTIMIZER = {"key": OPTIMIZER_KEY, **OPTIMIZER_REGISTRY[OPTIMIZER_KEY]}
    MODEL_LABEL = "modelB"
    PARAM_LIMIT = 100_000
    
    PROJECT_DIR = Path(__file__).resolve().parent
    DATASET_DIR = Path(args.dataset) if args.dataset else PROJECT_DIR / "dataset6"
    IMG_SIZE = (64, 64)
    BATCH_SIZE = args.batch_size
    SEED = args.seed
    EPOCHS = args.epochs
    AUG_MODE = args.aug
    USE_CACHE = not args.no_cache
    RESULTS_DIR = PROJECT_DIR / "results" / "modelB" / OPTIMIZER_KEY
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    COMMAND_LINE = "python " + " ".join(
        [Path(sys.argv[0]).name] + sys.argv[1:])
    print(f"\nCommand : {COMMAND_LINE}")
    print(f"Arguments: aug={AUG_MODE} epochs={EPOCHS} optimizer={OPTIMIZER_KEY} "
          f"batch={BATCH_SIZE} seed={SEED} cache={USE_CACHE}")
    (RESULTS_DIR / "command.txt").write_text(
        COMMAND_LINE + "\n" + json.dumps(vars(args), indent=2) + "\n",
        encoding="utf-8")
    
    class_names = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths, labels = [], []
    for class_name in class_names:
        for path in sorted((DATASET_DIR / class_name).rglob("*")):
            if path.is_file() and path.suffix.lower() in extensions:
                paths.append(str(path))
                labels.append(class_to_idx[class_name])
    
    if not paths:
        raise ValueError(f"No images found in {DATASET_DIR}")
    
    class_counts = Counter(labels)
    print("\nImages per class:")
    for name, idx in class_to_idx.items():
        print(f"{name:30s}: {class_counts[idx]}")
    print(f"\nTotal images: {len(paths)}")
    print(f"Number of classes: {num_classes}")
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
    
    print(f"\nTraining:   {len(X_train)} ({len(X_train)/len(paths):.1%})")
    print(f"Validation: {len(X_val)} ({len(X_val)/len(paths):.1%})")
    print(f"Testing:    {len(X_test)} ({len(X_test)/len(paths):.1%})")
    
    
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)  # downscale to 64x64
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    
    def build_augmentation():
        """Train-only augmentation (no trainable parameters, not part of the model).
        geometric: flips + mild rotation/zoom (colour is left untouched, because
        colour is a key cue for glass / vegetation / cardboard).
        full: adds translation, contrast and brightness jitter."""
        ops = [layers.RandomFlip("horizontal_and_vertical"),
               layers.RandomRotation(0.10, fill_mode="reflect"),
               layers.RandomZoom(0.10, fill_mode="reflect")]
        if AUG_MODE == "full":
            ops += [layers.RandomTranslation(0.10, 0.10, fill_mode="reflect"),
                    layers.RandomContrast(0.20),
                    layers.RandomBrightness(0.20, value_range=(0.0, 1.0))]
        return keras.Sequential(ops, name="train_augmentation")
    
    
    augmentation = build_augmentation()
    
    
    def make_dataset(paths_, labels_, training=False):
        ds = tf.data.Dataset.from_tensor_slices((paths_, labels_))
        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        if USE_CACHE:
            ds = ds.cache()  # cache the clean decoded images; augment afterwards
        if training:
            ds = ds.shuffle(len(paths_), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE)
        if training and AUG_MODE != "none":
            ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)
    
    
    train_ds = make_dataset(X_train, y_train, training=True)
    val_ds = make_dataset(X_val, y_val)
    test_ds = make_dataset(X_test, y_test)
    
    
    def sep_block(filters):
        """Depthwise 3x3 + pointwise 1x1 (SeparableConv2D) -> BN -> ReLU6."""
        return [layers.SeparableConv2D(filters, kernel_size=(3, 3), padding="same",
                                       use_bias=False),
                layers.BatchNormalization(),
                layers.ReLU(max_value=6.0)]
    
    
    def build_model():
        """modelB: depthwise-separable CNN, logits output.
    
        Hand-computed trainable parameters (BN counts gamma+beta = 2*C):
          stem   Conv2D 3x3 3->16            432  + BN 32           =    464
          S1     Sep 16->32     144+512=656       + BN 64           =    720
          S2     Sep 32->64    288+2048=2,336     + BN 128
                 Sep 64->64    576+4096=4,672     + BN 128          =  7,264
          S3     Sep 64->128  576+8192=8,768      + BN 256
                 Sep 128->128 1152+16384=17,536   + BN 256          = 26,816
          S4     Sep 128->256 1152+32768=33,920   + BN 512          = 34,432
          head   Dense 256->64 (16,448) + Dense 64->9 (585)         = 17,033
          TOTAL trainable                                           = 86,729
        (the script recomputes this from the real model and asserts <= 100,000)
        """
        return keras.Sequential([
            layers.Input(shape=(64, 64, 3)),
    
            layers.Conv2D(16, kernel_size=(3, 3), padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(max_value=6.0),
            layers.MaxPooling2D(pool_size=(2, 2)),          # 32x32x16
    
            *sep_block(32),
            layers.MaxPooling2D(pool_size=(2, 2)),          # 16x16x32
    
            *sep_block(64),
            *sep_block(64),
            layers.MaxPooling2D(pool_size=(2, 2)),          # 8x8x64
            layers.Dropout(0.10),
    
            *sep_block(128),
            *sep_block(128),
            layers.MaxPooling2D(pool_size=(2, 2)),          # 4x4x128
            layers.Dropout(0.10),
    
            *sep_block(256),                                # 4x4x256
    
            layers.GlobalAveragePooling2D(),                # 256
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.25),
            layers.Dense(num_classes),  # logits; softmax folded into the loss
        ], name="ModelB_Lightweight_SeparableCNN")
    def layer_breakdown(model):
        """Per-layer output shape, parameters (total / trainable) and MACs."""
        rows = []
        for layer in model.layers:
            out_shape = tuple(layer.output.shape[1:])
            in_shape = tuple(layer.input.shape[1:])
            trainable = sum(int(np.prod(w.shape)) for w in layer.trainable_weights)
            total = layer.count_params()
            macs = 0
            if isinstance(layer, layers.SeparableConv2D):
                k = layer.kernel_size[0] * layer.kernel_size[1]
                cin, cout = in_shape[-1], layer.filters
                macs = out_shape[0] * out_shape[1] * (k * cin + cin * cout)
            elif isinstance(layer, layers.Conv2D):
                k = layer.kernel_size[0] * layer.kernel_size[1]
                cin, cout = in_shape[-1], layer.filters
                macs = out_shape[0] * out_shape[1] * k * cin * cout
            elif isinstance(layer, layers.Dense):
                macs = in_shape[-1] * out_shape[-1]
            rows.append({
                "layer": layer.name,
                "type": layer.__class__.__name__,
                "output_shape": list(out_shape),
                "total_params": int(total),
                "trainable_params": int(trainable),
                "macs": int(macs),
            })
        return rows
    
    
    def format_breakdown(rows):
        header = f"{'Layer':28s}{'Type':22s}{'Output shape':18s}{'Params':>10s}{'Trainable':>12s}{'MACs':>14s}"
        lines = [header, "-" * len(header)]
        for r in rows:
            lines.append(
                f"{r['layer']:28s}{r['type']:22s}{str(tuple(r['output_shape'])):18s}"
                f"{r['total_params']:>10,}{r['trainable_params']:>12,}{r['macs']:>14,}")
        lines.append("-" * len(header))
        lines.append(
            f"{'TOTAL':28s}{'':22s}{'':18s}"
            f"{sum(r['total_params'] for r in rows):>10,}"
            f"{sum(r['trainable_params'] for r in rows):>12,}"
            f"{sum(r['macs'] for r in rows):>14,}")
        return "\n".join(lines)
    
    
    class EpochTimer(keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.epoch_times = []
    
        def on_epoch_begin(self, epoch, logs=None):
            self.t0 = time.perf_counter()
    
        def on_epoch_end(self, epoch, logs=None):
            self.epoch_times.append(float(time.perf_counter() - self.t0))
    
    
    def first_epoch_reaching(values, threshold, mode="ge"):
        """1-based epoch at which `values` first reaches the threshold, else None."""
        for i, v in enumerate(values):
            if (mode == "ge" and v >= threshold) or (mode == "le" and v <= threshold):
                return i + 1
        return None
    
    
    def run_experiment(opt_cfg):
        """Train a fresh modelB with one optimizer for EPOCHS epochs (no early
        stopping, so every run completes >= 20 epochs), restore the best
        validation-loss weights, then evaluate on the test set."""
        print(f"\n{'=' * 70}\nTraining {MODEL_LABEL} with {opt_cfg['label']}\n{'=' * 70}")
        keras.utils.set_random_seed(SEED)  # identical initial weights per optimizer
        model = build_model()
        model.compile(
            optimizer=opt_cfg["make"](),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )
    
        weights_path = RESULTS_DIR / f"best_{opt_cfg['key']}.weights.h5"
        timer = EpochTimer()
        callbacks = [
            keras.callbacks.ModelCheckpoint(
                filepath=str(weights_path), monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6,
                mode="min", verbose=1),
            timer,
        ]
    
        t0 = time.perf_counter()
        history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
                            callbacks=callbacks, verbose=2)
        total_time = time.perf_counter() - t0
    
        if weights_path.exists():
            model.load_weights(weights_path)  # best validation-loss weights
    
        test_loss, test_acc = model.evaluate(test_ds, verbose=0)
        logits = model.predict(test_ds, verbose=0)
        preds = logits.argmax(axis=1)
    
        hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        val_acc = history.history["val_accuracy"]
        best_vl = float(min(val_loss))
        last10 = val_loss[-10:]
        # The first epoch includes XLA/graph compilation, so the median (and the
        # mean excluding epoch 1) are the fair "time per epoch" figures.
        et = timer.epoch_times
        result = {
            "key": opt_cfg["key"],
            "label": opt_cfg["label"],
            "epochs_completed": len(val_loss),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "test_f1_macro": float(f1_score(y_test, preds, average="macro", zero_division=0)),
            "test_f1_weighted": float(f1_score(y_test, preds, average="weighted", zero_division=0)),
            "test_precision_macro": float(precision_score(y_test, preds, average="macro", zero_division=0)),
            "test_recall_macro": float(recall_score(y_test, preds, average="macro", zero_division=0)),
            "test_confusion_matrix": confusion_matrix(
                y_test, preds, labels=list(range(num_classes))).tolist(),
            "best_validation_loss": best_vl,
            "best_validation_loss_epoch": int(np.argmin(val_loss)) + 1,
            "best_validation_accuracy": float(max(val_acc)),
            "best_validation_accuracy_epoch": int(np.argmax(val_acc)) + 1,
            "final_training_loss": float(train_loss[-1]),
            "final_training_accuracy": float(history.history["accuracy"][-1]),
            "final_validation_loss": float(val_loss[-1]),
            "final_validation_accuracy": float(val_acc[-1]),
            "epochs_to_val_acc_60": first_epoch_reaching(val_acc, 0.60),
            "epochs_to_val_acc_70": first_epoch_reaching(val_acc, 0.70),
            "epochs_to_val_loss_1.0": first_epoch_reaching(val_loss, 1.0, "le"),
            "epochs_to_within_5pct_best_val_loss": first_epoch_reaching(
                val_loss, best_vl * 1.05, "le"),
            "train_loss_epoch_5": float(train_loss[4]) if len(train_loss) >= 5 else None,
            "train_loss_epoch_10": float(train_loss[9]) if len(train_loss) >= 10 else None,
            "val_loss_std_last10": float(np.std(last10)),
            "generalisation_gap_acc": float(
                history.history["accuracy"][-1] - val_acc[-1]),
            "total_training_time_seconds": float(total_time),
            "average_epoch_time_seconds": float(np.mean(et)),
            "average_epoch_time_excl_first_seconds": float(np.mean(et[1:])) if len(et) > 1 else float(np.mean(et)),
            "median_epoch_time_seconds": float(np.median(et)),
            "epoch_time_seconds": et,
            "history": hist,
        }
        print(f"{opt_cfg['label']}: test acc {test_acc:.4f} | macro F1 "
              f"{result['test_f1_macro']:.4f} | best val loss "
              f"{best_vl:.4f} @ epoch {result['best_validation_loss_epoch']} | "
              f"{result['median_epoch_time_seconds']:.2f}s/epoch (median)")
        return model, preds, result, weights_path
    
    
    keras.utils.set_random_seed(SEED)
    arch_model = build_model()
    arch_model.summary()
    rows = layer_breakdown(arch_model)
    breakdown_text = format_breakdown(rows)
    print("\n" + breakdown_text)
    
    trainable_params = sum(r["trainable_params"] for r in rows)
    total_params = sum(r["total_params"] for r in rows)
    non_trainable_params = total_params - trainable_params
    total_macs = sum(r["macs"] for r in rows)
    float32_size_kb = total_params * 4 / 1024
    
    print(f"\nTrainable parameters:     {trainable_params:,}")
    print(f"Non-trainable parameters: {non_trainable_params:,} (BatchNorm moving statistics)")
    print(f"Total parameters:         {total_params:,}")
    print(f"MACs per inference:       {total_macs:,}")
    print(f"Float32 weight size:      {float32_size_kb:.1f} KB")
    
    assert trainable_params <= PARAM_LIMIT, (
        f"{MODEL_LABEL} exceeds the {PARAM_LIMIT:,} trainable-parameter "
        f"limit: {trainable_params:,}")
    print(f"Parameter constraint satisfied: {trainable_params:,} <= {PARAM_LIMIT:,} "
          f"({trainable_params / PARAM_LIMIT:.1%} of budget).")
    
    (RESULTS_DIR / "layer_breakdown.txt").write_text(breakdown_text, encoding="utf-8")
    (RESULTS_DIR / "layer_breakdown.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")
    del arch_model
    
    main_model, main_preds, main_res, weights_path = run_experiment(OPTIMIZER)
    weights_path.unlink(missing_ok=True)
    model_path = RESULTS_DIR / "model.keras"
    main_model.save(model_path)
    disk_size_kb = model_path.stat().st_size / 1024
    print(f"\nSaved model file size: {disk_size_kb:.1f} KB")
    
    report_dict = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, output_dict=True, zero_division=0)
    report_text = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, zero_division=0)
    print("\n" + report_text)
    
    cm = confusion_matrix(y_test, main_preds, labels=list(range(num_classes)))
    
    final_metrics = {
        "command": COMMAND_LINE,
        "model": MODEL_LABEL,
        "augmentation": AUG_MODE,
        "optimizer": main_res["label"],
        "test_loss": main_res["test_loss"],
        "test_accuracy": main_res["test_accuracy"],
        "test_precision_macro": main_res["test_precision_macro"],
        "test_recall_macro": main_res["test_recall_macro"],
        "test_f1_macro": main_res["test_f1_macro"],
        "test_f1_weighted": main_res["test_f1_weighted"],
        "test_classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
        "trainable_parameters": int(trainable_params),
        "non_trainable_parameters": int(non_trainable_params),
        "total_parameters": int(total_params),
        "parameter_limit": PARAM_LIMIT,
        "macs_per_inference": int(total_macs),
        "float32_weight_size_kb": float(float32_size_kb),
        "model_file_size_kb": float(disk_size_kb),
        "epochs_completed": main_res["epochs_completed"],
        "best_validation_loss": main_res["best_validation_loss"],
        "best_validation_loss_epoch": main_res["best_validation_loss_epoch"],
        "best_validation_accuracy": main_res["best_validation_accuracy"],
        "best_validation_accuracy_epoch": main_res["best_validation_accuracy_epoch"],
        "final_training_loss": main_res["final_training_loss"],
        "final_training_accuracy": main_res["final_training_accuracy"],
        "final_validation_loss": main_res["final_validation_loss"],
        "final_validation_accuracy": main_res["final_validation_accuracy"],
        "total_training_time_seconds": main_res["total_training_time_seconds"],
        "average_epoch_time_seconds": main_res["average_epoch_time_seconds"],
        "average_epoch_time_excl_first_seconds": main_res["average_epoch_time_excl_first_seconds"],
        "median_epoch_time_seconds": main_res["median_epoch_time_seconds"],
    }
    (RESULTS_DIR / "metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
    (RESULTS_DIR / "training_history.json").write_text(
        json.dumps({**main_res["history"], "epoch_time_seconds": main_res["epoch_time_seconds"]}, indent=2),
        encoding="utf-8")
    (RESULTS_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
    np.savetxt(RESULTS_DIR / "confusion_matrix.csv", cm, delimiter=",", fmt="%d")
    
    (RESULTS_DIR / "run_metadata.json").write_text(json.dumps({
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "command": COMMAND_LINE,
        "arguments": vars(args),
        "model": MODEL_LABEL,
        "dataset": str(DATASET_DIR),
        "image_size": list(IMG_SIZE),
        "batch_size": BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "seed": SEED,
        "optimizer": main_res["label"],
        "loss": "SparseCategoricalCrossentropy(from_logits=True)",
        "augmentation": AUG_MODE,
        "lr_schedule": "ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6)",
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "num_images": len(paths),
        "train_images": len(X_train),
        "validation_images": len(X_val),
        "test_images": len(X_test),
    }, indent=2), encoding="utf-8")
    
    plt.figure(figsize=(12, 6))
    plt.bar(class_names, [class_counts[class_to_idx[n]] for n in class_names])
    plt.title("Images per Class")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
    plt.close()
    
    epochs_axis = range(1, main_res["epochs_completed"] + 1)
    h = main_res["history"]
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, h["loss"], label="Training Loss")
    plt.plot(epochs_axis, h["val_loss"], label="Validation Loss")
    plt.title(f"{MODEL_LABEL}: Training and Validation Loss ({main_res['label']})")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=150)
    plt.close()
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, h["accuracy"], label="Training Accuracy")
    plt.plot(epochs_axis, h["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{MODEL_LABEL}: Training and Validation Accuracy ({main_res['label']})")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "accuracy_curve.png", dpi=150)
    plt.close()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
    ax.set_title(f"{MODEL_LABEL}: Test Confusion Matrix ({main_res['label']})")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()
    
    print(f"\nResults saved to: {RESULTS_DIR}")


## 4. Custom-model training and evaluation

The optimizer cells above satisfy the minimum 20-epoch training requirement. Each result is displayed immediately below its training cell with training/validation curves, a confusion matrix, a per-class classification report, accuracy, precision, recall, F1 score, parameter count, and elapsed training time.

After all custom-model runs are complete, use the comparison cell below to identify the best current optimizer/model combination. The final discussion should be updated from those displayed results if a rerun changes the ranking.


## Interpretation of custom-model results

The displayed results quantify the accuracy-resource trade-off between the standard and lightweight designs. Use macro precision, macro recall, and macro F1 alongside accuracy because the class supports are not identical. Inspect the confusion matrices and per-class reports to identify difficult categories such as visually similar waste types.


## 5. Lightweight state-of-the-art networks

MobileNetV2 and EfficientNet-B0 are fine-tuned using the same data split as the custom models. Each cell trains the classifier head and then fine-tunes the top backbone layers, displaying the combined learning curves, classification report, confusion matrix, parameter count, and test metrics inline. ImageNet weights are used by default; internet access may be required the first time the weights are downloaded.


## Pretrained lightweight networks — MobileNetV2 and EfficientNet-B0

In [ ]:
def build_sota(model_key, model_res=128, weights='imagenet', dropout=.2):
    if model_key == 'mobilenetv2':
        backbone = keras.applications.MobileNetV2(input_shape=(model_res, model_res, 3), include_top=False, weights=weights)
        preprocess = layers.Rescaling(2.0, offset=-1.0)
    elif model_key == 'efficientnetb0':
        backbone = keras.applications.EfficientNetB0(input_shape=(model_res, model_res, 3), include_top=False, weights=weights)
        preprocess = layers.Rescaling(255.0)
    else: raise ValueError(model_key)
    inputs = keras.Input((64, 64, 3))
    x = layers.Resizing(model_res, model_res)(inputs)
    x = preprocess(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return keras.Model(inputs, layers.Dense(len(class_names))(x), name=model_key), backbone

def train_sota(model_key, head_epochs=8, fine_epochs=30, fine_tune_layers=60, fine_lr=1e-4):
    keras.utils.set_random_seed(SEED)
    model, backbone = build_sota(model_key)
    backbone.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    start = time.perf_counter()
    head_history = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs, callbacks=[keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=.5, min_lr=1e-7)], verbose=2)
    backbone.trainable = True
    for layer in backbone.layers[:-fine_tune_layers]: layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization): layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(fine_lr), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    fine_history = model.fit(train_ds, validation_data=val_ds, initial_epoch=head_epochs, epochs=head_epochs+fine_epochs, callbacks=[keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=.5, min_lr=1e-7)], verbose=2)
    history = {key: head_history.history.get(key, []) + fine_history.history.get(key, []) for key in set(head_history.history) | set(fine_history.history)}
    h = keras.callbacks.History(); h.history = history
    result = inline_result(model, h, f'{model_key} fine-tuning', time.perf_counter()-start)
    NOTEBOOK_RESULTS[model_key] = result
    return result


### `python mobilenet.py`

In [ ]:
MOBILENETV2_RESULT = train_sota('mobilenetv2', fine_tune_layers=60, fine_lr=1e-4)


### `python efficientnet.py`

In [ ]:
EFFICIENTNETB0_RESULT = train_sota('efficientnetb0', fine_tune_layers=80, fine_lr=5e-5)


In [ ]:
if False:
    import os
    import sys
    import json
    import time
    import argparse
    from pathlib import Path
    from datetime import datetime, timezone
    from collections import Counter
    
    import numpy as np
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import matplotlib.pyplot as plt
    
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        ConfusionMatrixDisplay,
        f1_score,
        precision_score,
        recall_score,
    )
    MODEL_KEY = "mobilenetv2"
    MODEL_LABEL = "MobileNetV2"
    ALLOWED_SIZES = {96, 128, 160, 192, 224}
    
    parser = argparse.ArgumentParser(
        description=f"Fine-tune ImageNet-pretrained {MODEL_LABEL}.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument("--img-size", type=int, default=128,
                        help="resolution the 64x64 images are upsampled to inside the model")
    parser.add_argument("--head-epochs", type=int, default=8,
                        help="phase 1: epochs with the backbone frozen")
    parser.add_argument("--fine-epochs", type=int, default=30,
                        help="phase 2: epochs with the top backbone layers unfrozen")
    parser.add_argument("--fine-tune-layers", type=int, default=60,
                        help="number of backbone layers (from the top) to unfreeze in phase 2")
    parser.add_argument("--head-lr", type=float, default=1e-3, help="Adam lr, phase 1")
    parser.add_argument("--fine-lr", type=float, default=1e-4, help="Adam lr, phase 2")
    parser.add_argument("--dropout", type=float, default=0.2, help="dropout before the classifier")
    parser.add_argument("--weights", choices=["imagenet", "none"], default="imagenet",
                        help="'none' = random initialisation (no pretraining)")
    parser.add_argument("--aug", choices=["none", "geometric", "full"],
                        default=os.environ.get("AUG_MODE", "none"),
                        help="training-only augmentation (keep equal to modelA/modelB)")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--dataset", default=os.environ.get("DATASET_DIR"),
                        help="dataset root (one sub-folder per class); "
                             "default: <script dir>/dataset6")
    parser.add_argument("--no-cache", action="store_true",
                        help="do not cache decoded images in RAM")
    args = parser.parse_args()
    
    if args.head_epochs < 1 or args.fine_epochs < 1:
        parser.error("--head-epochs and --fine-epochs must both be >= 1")
    if args.fine_tune_layers < 1:
        parser.error("--fine-tune-layers must be at least 1")
    if ALLOWED_SIZES and args.weights == "imagenet" and args.img_size not in ALLOWED_SIZES:
        parser.error(f"{MODEL_LABEL} ImageNet weights support --img-size in "
                     f"{sorted(ALLOWED_SIZES)} (got {args.img_size})")
    TOTAL_EPOCHS = args.head_epochs + args.fine_epochs
    if TOTAL_EPOCHS < 20:
        print(f"WARNING: {TOTAL_EPOCHS} total epochs is below the assignment minimum of 20.")
    
    PROJECT_DIR = Path(__file__).resolve().parent
    DATASET_DIR = Path(args.dataset) if args.dataset else PROJECT_DIR / "dataset6"
    IMG_SIZE = (64, 64)
    MODEL_RES = args.img_size
    BATCH_SIZE = args.batch_size
    SEED = args.seed
    AUG_MODE = args.aug
    USE_CACHE = not args.no_cache
    WEIGHTS = None if args.weights == "none" else "imagenet"
    RESULTS_DIR = PROJECT_DIR / "results" / MODEL_KEY
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    COMMAND_LINE = "python " + " ".join([Path(sys.argv[0]).name] + sys.argv[1:])
    print(f"\nCommand : {COMMAND_LINE}")
    print(f"Arguments: model={MODEL_KEY} img_size={MODEL_RES} head_epochs={args.head_epochs} "
          f"fine_epochs={args.fine_epochs} fine_tune_layers={args.fine_tune_layers} "
          f"weights={args.weights} aug={AUG_MODE} batch={BATCH_SIZE} seed={SEED}")
    (RESULTS_DIR / "command.txt").write_text(
        COMMAND_LINE + "\n" + json.dumps(vars(args), indent=2) + "\n", encoding="utf-8")
    class_names = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths, labels = [], []
    for class_name in class_names:
        for path in sorted((DATASET_DIR / class_name).rglob("*")):
            if path.is_file() and path.suffix.lower() in extensions:
                paths.append(str(path))
                labels.append(class_to_idx[class_name])
    
    if not paths:
        raise ValueError(f"No images found in {DATASET_DIR}")
    
    class_counts = Counter(labels)
    print("\nImages per class:")
    for name, idx in class_to_idx.items():
        print(f"{name:30s}: {class_counts[idx]}")
    print(f"\nTotal images: {len(paths)}")
    print(f"Number of classes: {num_classes}")
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
    
    print(f"\nTraining:   {len(X_train)} ({len(X_train)/len(paths):.1%})")
    print(f"Validation: {len(X_val)} ({len(X_val)/len(paths):.1%})")
    print(f"Testing:    {len(X_test)} ({len(X_test)/len(paths):.1%})")
    
    
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    
    def build_augmentation():
        """Train-only augmentation (no trainable parameters, not part of the model)."""
        ops = [layers.RandomFlip("horizontal_and_vertical"),
               layers.RandomRotation(0.10, fill_mode="reflect"),
               layers.RandomZoom(0.10, fill_mode="reflect")]
        if AUG_MODE == "full":
            ops += [layers.RandomTranslation(0.10, 0.10, fill_mode="reflect"),
                    layers.RandomContrast(0.20),
                    layers.RandomBrightness(0.20, value_range=(0.0, 1.0))]
        return keras.Sequential(ops, name="train_augmentation")
    
    
    augmentation = build_augmentation()
    
    
    def make_dataset(paths_, labels_, training=False):
        ds = tf.data.Dataset.from_tensor_slices((paths_, labels_))
        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        if USE_CACHE:
            ds = ds.cache()
        if training:
            ds = ds.shuffle(len(paths_), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE)
        if training and AUG_MODE != "none":
            ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)
    
    
    train_ds = make_dataset(X_train, y_train, training=True)
    val_ds = make_dataset(X_val, y_val)
    test_ds = make_dataset(X_test, y_test)
    def build_model():
        """64x64 [0,1] image -> upsample -> backbone preprocessing -> pretrained
        backbone -> GAP -> Dropout -> Dense(num_classes) logits."""
        backbone = keras.applications.MobileNetV2(
            input_shape=(MODEL_RES, MODEL_RES, 3), include_top=False,
            weights=WEIGHTS, alpha=1.0)
        inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="image_64x64")
        x = layers.Resizing(MODEL_RES, MODEL_RES, interpolation="bilinear",
                            name="upsample")(inputs)
        x = layers.Rescaling(2.0, offset=-1.0, name="mobilenetv2_preprocess")(x)
        x = backbone(x, training=False)
        x = layers.GlobalAveragePooling2D(name="gap")(x)
        x = layers.Dropout(args.dropout, name="head_dropout")(x)
        outputs = layers.Dense(num_classes, name="classifier")(x)
        return keras.Model(inputs, outputs, name=f"{MODEL_KEY}_finetuned"), backbone
    
    
    def set_fine_tune(backbone, n_layers):
        """Unfreeze the top n_layers of the backbone; BatchNorm layers stay frozen."""
        backbone.trainable = True
        for layer in backbone.layers[:-n_layers]:
            layer.trainable = False
        for layer in backbone.layers:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False
    
    
    
    def count_trainable(model):
        return int(sum(int(np.prod(w.shape)) for w in model.trainable_weights))
    
    
    def count_macs(model):
        """Approximate MACs per inference: Conv2D, DepthwiseConv2D, SeparableConv2D
        and Dense layers (same convention as modelA/modelB). Walks into the backbone."""
        total = 0
        skipped = 0
    
        def walk(m):
            nonlocal total, skipped
            for layer in m.layers:
                if isinstance(layer, keras.Model):
                    walk(layer)
                    continue
                try:
                    out_shape = tuple(layer.output.shape[1:])
                    in_shape = tuple(layer.input.shape[1:])
                except Exception:
                    skipped += 1
                    continue
                if isinstance(layer, layers.DepthwiseConv2D):
                    k = layer.kernel_size[0] * layer.kernel_size[1]
                    total += out_shape[0] * out_shape[1] * k * out_shape[-1]
                elif isinstance(layer, layers.SeparableConv2D):
                    k = layer.kernel_size[0] * layer.kernel_size[1]
                    cin, cout = in_shape[-1], layer.filters
                    total += out_shape[0] * out_shape[1] * (k * cin + cin * cout)
                elif isinstance(layer, layers.Conv2D):
                    k = layer.kernel_size[0] * layer.kernel_size[1]
                    total += out_shape[0] * out_shape[1] * k * in_shape[-1] * layer.filters
                elif isinstance(layer, layers.Dense):
                    total += in_shape[-1] * out_shape[-1]
    
        walk(model)
        return int(total), skipped
    
    
    class EpochTimer(keras.callbacks.Callback):
        """Records every epoch across both training phases."""
        def __init__(self):
            super().__init__()
            self.epoch_times = []
    
        def on_epoch_begin(self, epoch, logs=None):
            self.t0 = time.perf_counter()
    
        def on_epoch_end(self, epoch, logs=None):
            self.epoch_times.append(float(time.perf_counter() - self.t0))
    
    
    class BestWeights(keras.callbacks.Callback):
        """Keeps the weights with the lowest validation loss across BOTH phases."""
        def __init__(self):
            super().__init__()
            self.best = float("inf")
            self.weights = None
            self.epoch = None
            self.seen = 0
    
        def on_epoch_end(self, epoch, logs=None):
            self.seen += 1
            val = (logs or {}).get("val_loss")
            if val is not None and val < self.best:
                self.best = float(val)
                self.weights = self.model.get_weights()
                self.epoch = self.seen
    
    
    def compile_model(model, lr):
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )
    
    
    def lr_callback():
        return keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7,
            mode="min", verbose=1)
    keras.utils.set_random_seed(SEED)
    model, backbone = build_model()
    model.summary()
    summary_lines = []
    model.summary(print_fn=lambda s, **kw: summary_lines.append(s), expand_nested=False)
    (RESULTS_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
    
    total_params = int(model.count_params())
    backbone_params = int(backbone.count_params())
    head_params = total_params - backbone_params
    macs, macs_skipped = count_macs(model)
    float32_size_mb = total_params * 4 / (1024 ** 2)
    print(f"\nBackbone parameters:      {backbone_params:,}")
    print(f"Head parameters:          {head_params:,}")
    print(f"Total parameters:         {total_params:,}")
    print(f"MACs per inference:       {macs:,} at {MODEL_RES}x{MODEL_RES} input "
          f"(approx.; conv + dense layers; {macs_skipped} layers skipped)")
    print(f"Float32 weight size:      {float32_size_mb:.2f} MB")
    timer, best = EpochTimer(), BestWeights()
    backbone.trainable = False
    compile_model(model, args.head_lr)
    head_trainable = count_trainable(model)
    print(f"\n{'=' * 70}\nPhase 1: head only (backbone frozen), "
          f"{head_trainable:,} trainable parameters, lr={args.head_lr}\n{'=' * 70}")
    t0 = time.perf_counter()
    head_history = model.fit(train_ds, validation_data=val_ds,
                             epochs=args.head_epochs,
                             callbacks=[lr_callback(), best, timer], verbose=2)
    set_fine_tune(backbone, args.fine_tune_layers)
    compile_model(model, args.fine_lr)
    fine_trainable = count_trainable(model)
    print(f"\n{'=' * 70}\nPhase 2: fine-tuning top {args.fine_tune_layers} backbone layers, "
          f"{fine_trainable:,} trainable parameters, lr={args.fine_lr}\n{'=' * 70}")
    fine_history = model.fit(train_ds, validation_data=val_ds,
                             initial_epoch=args.head_epochs, epochs=TOTAL_EPOCHS,
                             callbacks=[lr_callback(), best, timer], verbose=2)
    total_time = time.perf_counter() - t0
    
    if best.weights is not None:
        model.set_weights(best.weights)
    print(f"\nRestored best weights: val loss {best.best:.4f} @ epoch {best.epoch}")
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    logits = model.predict(test_ds, verbose=0)
    main_preds = logits.argmax(axis=1)
    
    history = {
        key: [float(v) for v in head_history.history.get(key, [])]
        + [float(v) for v in fine_history.history.get(key, [])]
        for key in set(head_history.history) | set(fine_history.history)
    }
    n_epochs = len(history["val_loss"])
    train_loss, val_loss = history["loss"], history["val_loss"]
    val_acc, train_acc = history["val_accuracy"], history["accuracy"]
    et = timer.epoch_times
    fine_et = et[args.head_epochs:]
    
    model_path = RESULTS_DIR / "model.keras"
    model.save(model_path)
    disk_size_kb = model_path.stat().st_size / 1024
    disk_size_mb = disk_size_kb / 1024
    print(f"\nSaved model file size: {disk_size_mb:.2f} MB ({disk_size_kb:.1f} KB)")
    
    report_dict = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, output_dict=True, zero_division=0)
    report_text = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, zero_division=0)
    print("\n" + report_text)
    cm = confusion_matrix(y_test, main_preds, labels=list(range(num_classes)))
    
    final_metrics = {
        "command": COMMAND_LINE,
        "model": MODEL_LABEL,
        "weights": args.weights,
        "backbone_input_resolution": MODEL_RES,
        "augmentation": AUG_MODE,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "test_precision_macro": float(precision_score(y_test, main_preds, average="macro", zero_division=0)),
        "test_recall_macro": float(recall_score(y_test, main_preds, average="macro", zero_division=0)),
        "test_f1_macro": float(f1_score(y_test, main_preds, average="macro", zero_division=0)),
        "test_f1_weighted": float(f1_score(y_test, main_preds, average="weighted", zero_division=0)),
        "test_classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
        "total_parameters": total_params,
        "backbone_parameters": backbone_params,
        "head_parameters": head_params,
        "trainable_parameters_phase1": head_trainable,
        "trainable_parameters_phase2": fine_trainable,
        "macs_per_inference": macs,
        "float32_weight_size_mb": float(float32_size_mb),
        "model_file_size_kb": float(disk_size_kb),
        "model_file_size_mb": float(disk_size_mb),
        "epochs_completed": n_epochs,
        "best_validation_loss": float(best.best),
        "best_validation_loss_epoch": int(best.epoch),
        "best_validation_accuracy": float(max(val_acc)),
        "best_validation_accuracy_epoch": int(np.argmax(val_acc)) + 1,
        "final_training_loss": float(train_loss[-1]),
        "final_training_accuracy": float(train_acc[-1]),
        "final_validation_loss": float(val_loss[-1]),
        "final_validation_accuracy": float(val_acc[-1]),
        "total_training_time_seconds": float(total_time),
        "average_epoch_time_seconds": float(np.mean(et)),
        "average_epoch_time_excl_first_seconds": float(np.mean(et[1:])) if len(et) > 1 else float(np.mean(et)),
        "median_epoch_time_seconds": float(np.median(et)),
        "median_epoch_time_fine_tune_seconds": float(np.median(fine_et)) if fine_et else None,
    }
    (RESULTS_DIR / "metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
    (RESULTS_DIR / "training_history.json").write_text(
        json.dumps({**history, "epoch_time_seconds": et}, indent=2), encoding="utf-8")
    (RESULTS_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
    np.savetxt(RESULTS_DIR / "confusion_matrix.csv", cm, delimiter=",", fmt="%d")
    
    (RESULTS_DIR / "run_metadata.json").write_text(json.dumps({
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "command": COMMAND_LINE,
        "arguments": vars(args),
        "model": MODEL_LABEL,
        "dataset": str(DATASET_DIR),
        "data_pipeline_image_size": list(IMG_SIZE),
        "backbone_input_resolution": MODEL_RES,
        "batch_size": BATCH_SIZE,
        "head_epochs": args.head_epochs,
        "fine_tune_epochs": args.fine_epochs,
        "fine_tune_layers": args.fine_tune_layers,
        "optimizer": f"Adam (head lr={args.head_lr}, fine-tune lr={args.fine_lr})",
        "loss": "SparseCategoricalCrossentropy(from_logits=True)",
        "lr_schedule": "ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7) per phase",
        "seed": SEED,
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "num_images": len(paths),
        "train_images": len(X_train),
        "validation_images": len(X_val),
        "test_images": len(X_test),
    }, indent=2), encoding="utf-8")
    
    print(f"\n{MODEL_LABEL}: test acc {test_acc:.4f} | macro F1 "
          f"{final_metrics['test_f1_macro']:.4f} | params {total_params:,} | "
          f"{disk_size_mb:.2f} MB | {final_metrics['median_epoch_time_seconds']:.2f}s/epoch (median)")
    plt.figure(figsize=(12, 6))
    plt.bar(class_names, [class_counts[class_to_idx[n]] for n in class_names])
    plt.title("Images per Class")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
    plt.close()
    
    epochs_axis = range(1, n_epochs + 1)
    boundary = args.head_epochs + 0.5
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, train_loss, label="Training Loss")
    plt.plot(epochs_axis, val_loss, label="Validation Loss")
    plt.axvline(boundary, color="gray", linestyle="--", label="unfreeze backbone")
    plt.title(f"{MODEL_LABEL}: Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=150)
    plt.close()
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, train_acc, label="Training Accuracy")
    plt.plot(epochs_axis, val_acc, label="Validation Accuracy")
    plt.axvline(boundary, color="gray", linestyle="--", label="unfreeze backbone")
    plt.title(f"{MODEL_LABEL}: Training and Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "accuracy_curve.png", dpi=150)
    plt.close()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
    ax.set_title(f"{MODEL_LABEL}: Test Confusion Matrix")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()
    
    print(f"\nResults saved to: {RESULTS_DIR}")


In [ ]:
if False:
    import os
    import sys
    import json
    import time
    import argparse
    from pathlib import Path
    from datetime import datetime, timezone
    from collections import Counter
    
    import numpy as np
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import matplotlib.pyplot as plt
    
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        ConfusionMatrixDisplay,
        f1_score,
        precision_score,
        recall_score,
    )
    MODEL_KEY = "efficientnetb0"
    MODEL_LABEL = "EfficientNet-B0"
    ALLOWED_SIZES = None
    
    parser = argparse.ArgumentParser(
        description=f"Fine-tune ImageNet-pretrained {MODEL_LABEL}.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument("--img-size", type=int, default=128,
                        help="resolution the 64x64 images are upsampled to inside the model")
    parser.add_argument("--head-epochs", type=int, default=8,
                        help="phase 1: epochs with the backbone frozen")
    parser.add_argument("--fine-epochs", type=int, default=30,
                        help="phase 2: epochs with the top backbone layers unfrozen")
    parser.add_argument("--fine-tune-layers", type=int, default=80,
                        help="number of backbone layers (from the top) to unfreeze in phase 2")
    parser.add_argument("--head-lr", type=float, default=1e-3, help="Adam lr, phase 1")
    parser.add_argument("--fine-lr", type=float, default=5e-5, help="Adam lr, phase 2")
    parser.add_argument("--dropout", type=float, default=0.2, help="dropout before the classifier")
    parser.add_argument("--weights", choices=["imagenet", "none"], default="imagenet",
                        help="'none' = random initialisation (no pretraining)")
    parser.add_argument("--aug", choices=["none", "geometric", "full"],
                        default=os.environ.get("AUG_MODE", "none"),
                        help="training-only augmentation (keep equal to modelA/modelB)")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--dataset", default=os.environ.get("DATASET_DIR"),
                        help="dataset root (one sub-folder per class); "
                             "default: <script dir>/dataset6")
    parser.add_argument("--no-cache", action="store_true",
                        help="do not cache decoded images in RAM")
    args = parser.parse_args()
    
    if args.head_epochs < 1 or args.fine_epochs < 1:
        parser.error("--head-epochs and --fine-epochs must both be >= 1")
    if args.fine_tune_layers < 1:
        parser.error("--fine-tune-layers must be at least 1")
    if ALLOWED_SIZES and args.weights == "imagenet" and args.img_size not in ALLOWED_SIZES:
        parser.error(f"{MODEL_LABEL} ImageNet weights support --img-size in "
                     f"{sorted(ALLOWED_SIZES)} (got {args.img_size})")
    TOTAL_EPOCHS = args.head_epochs + args.fine_epochs
    if TOTAL_EPOCHS < 20:
        print(f"WARNING: {TOTAL_EPOCHS} total epochs is below the assignment minimum of 20.")
    
    PROJECT_DIR = Path(__file__).resolve().parent
    DATASET_DIR = Path(args.dataset) if args.dataset else PROJECT_DIR / "dataset6"
    IMG_SIZE = (64, 64)
    MODEL_RES = args.img_size
    BATCH_SIZE = args.batch_size
    SEED = args.seed
    AUG_MODE = args.aug
    USE_CACHE = not args.no_cache
    WEIGHTS = None if args.weights == "none" else "imagenet"
    RESULTS_DIR = PROJECT_DIR / "results" / MODEL_KEY
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    COMMAND_LINE = "python " + " ".join([Path(sys.argv[0]).name] + sys.argv[1:])
    print(f"\nCommand : {COMMAND_LINE}")
    print(f"Arguments: model={MODEL_KEY} img_size={MODEL_RES} head_epochs={args.head_epochs} "
          f"fine_epochs={args.fine_epochs} fine_tune_layers={args.fine_tune_layers} "
          f"weights={args.weights} aug={AUG_MODE} batch={BATCH_SIZE} seed={SEED}")
    (RESULTS_DIR / "command.txt").write_text(
        COMMAND_LINE + "\n" + json.dumps(vars(args), indent=2) + "\n", encoding="utf-8")
    class_names = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths, labels = [], []
    for class_name in class_names:
        for path in sorted((DATASET_DIR / class_name).rglob("*")):
            if path.is_file() and path.suffix.lower() in extensions:
                paths.append(str(path))
                labels.append(class_to_idx[class_name])
    
    if not paths:
        raise ValueError(f"No images found in {DATASET_DIR}")
    
    class_counts = Counter(labels)
    print("\nImages per class:")
    for name, idx in class_to_idx.items():
        print(f"{name:30s}: {class_counts[idx]}")
    print(f"\nTotal images: {len(paths)}")
    print(f"Number of classes: {num_classes}")
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
    
    print(f"\nTraining:   {len(X_train)} ({len(X_train)/len(paths):.1%})")
    print(f"Validation: {len(X_val)} ({len(X_val)/len(paths):.1%})")
    print(f"Testing:    {len(X_test)} ({len(X_test)/len(paths):.1%})")
    
    
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    
    def build_augmentation():
        """Train-only augmentation (no trainable parameters, not part of the model)."""
        ops = [layers.RandomFlip("horizontal_and_vertical"),
               layers.RandomRotation(0.10, fill_mode="reflect"),
               layers.RandomZoom(0.10, fill_mode="reflect")]
        if AUG_MODE == "full":
            ops += [layers.RandomTranslation(0.10, 0.10, fill_mode="reflect"),
                    layers.RandomContrast(0.20),
                    layers.RandomBrightness(0.20, value_range=(0.0, 1.0))]
        return keras.Sequential(ops, name="train_augmentation")
    
    
    augmentation = build_augmentation()
    
    
    def make_dataset(paths_, labels_, training=False):
        ds = tf.data.Dataset.from_tensor_slices((paths_, labels_))
        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        if USE_CACHE:
            ds = ds.cache()
        if training:
            ds = ds.shuffle(len(paths_), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE)
        if training and AUG_MODE != "none":
            ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)
    
    
    train_ds = make_dataset(X_train, y_train, training=True)
    val_ds = make_dataset(X_val, y_val)
    test_ds = make_dataset(X_test, y_test)
    def build_model():
        """64x64 [0,1] image -> upsample -> backbone preprocessing -> pretrained
        backbone -> GAP -> Dropout -> Dense(num_classes) logits."""
        backbone = keras.applications.EfficientNetB0(
            input_shape=(MODEL_RES, MODEL_RES, 3), include_top=False,
            weights=WEIGHTS)
        inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="image_64x64")
        x = layers.Resizing(MODEL_RES, MODEL_RES, interpolation="bilinear",
                            name="upsample")(inputs)
        has_builtin_preprocessing = any(
            isinstance(l, layers.Rescaling) for l in backbone.layers[:6])
        if has_builtin_preprocessing:
            x = layers.Rescaling(255.0, name="to_0_255")(x)
        else:
            x = layers.Normalization(
                mean=[0.485, 0.456, 0.406],
                variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2],
                name="imagenet_norm")(x)
        x = backbone(x, training=False)
        x = layers.GlobalAveragePooling2D(name="gap")(x)
        x = layers.Dropout(args.dropout, name="head_dropout")(x)
        outputs = layers.Dense(num_classes, name="classifier")(x)
        return keras.Model(inputs, outputs, name=f"{MODEL_KEY}_finetuned"), backbone
    
    
    def set_fine_tune(backbone, n_layers):
        """Unfreeze the top n_layers of the backbone; BatchNorm layers stay frozen."""
        backbone.trainable = True
        for layer in backbone.layers[:-n_layers]:
            layer.trainable = False
        for layer in backbone.layers:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False
    
    
    
    def count_trainable(model):
        return int(sum(int(np.prod(w.shape)) for w in model.trainable_weights))
    
    
    def count_macs(model):
        """Approximate MACs per inference: Conv2D, DepthwiseConv2D, SeparableConv2D
        and Dense layers (same convention as modelA/modelB). Walks into the backbone."""
        total = 0
        skipped = 0
    
        def walk(m):
            nonlocal total, skipped
            for layer in m.layers:
                if isinstance(layer, keras.Model):
                    walk(layer)
                    continue
                try:
                    out_shape = tuple(layer.output.shape[1:])
                    in_shape = tuple(layer.input.shape[1:])
                except Exception:
                    skipped += 1
                    continue
                if isinstance(layer, layers.DepthwiseConv2D):
                    k = layer.kernel_size[0] * layer.kernel_size[1]
                    total += out_shape[0] * out_shape[1] * k * out_shape[-1]
                elif isinstance(layer, layers.SeparableConv2D):
                    k = layer.kernel_size[0] * layer.kernel_size[1]
                    cin, cout = in_shape[-1], layer.filters
                    total += out_shape[0] * out_shape[1] * (k * cin + cin * cout)
                elif isinstance(layer, layers.Conv2D):
                    k = layer.kernel_size[0] * layer.kernel_size[1]
                    total += out_shape[0] * out_shape[1] * k * in_shape[-1] * layer.filters
                elif isinstance(layer, layers.Dense):
                    total += in_shape[-1] * out_shape[-1]
    
        walk(model)
        return int(total), skipped
    
    
    class EpochTimer(keras.callbacks.Callback):
        """Records every epoch across both training phases."""
        def __init__(self):
            super().__init__()
            self.epoch_times = []
    
        def on_epoch_begin(self, epoch, logs=None):
            self.t0 = time.perf_counter()
    
        def on_epoch_end(self, epoch, logs=None):
            self.epoch_times.append(float(time.perf_counter() - self.t0))
    
    
    class BestWeights(keras.callbacks.Callback):
        """Keeps the weights with the lowest validation loss across BOTH phases."""
        def __init__(self):
            super().__init__()
            self.best = float("inf")
            self.weights = None
            self.epoch = None
            self.seen = 0
    
        def on_epoch_end(self, epoch, logs=None):
            self.seen += 1
            val = (logs or {}).get("val_loss")
            if val is not None and val < self.best:
                self.best = float(val)
                self.weights = self.model.get_weights()
                self.epoch = self.seen
    
    
    def compile_model(model, lr):
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )
    
    
    def lr_callback():
        return keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7,
            mode="min", verbose=1)
    keras.utils.set_random_seed(SEED)
    model, backbone = build_model()
    model.summary()
    summary_lines = []
    model.summary(print_fn=lambda s, **kw: summary_lines.append(s), expand_nested=False)
    (RESULTS_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
    
    total_params = int(model.count_params())
    backbone_params = int(backbone.count_params())
    head_params = total_params - backbone_params
    macs, macs_skipped = count_macs(model)
    float32_size_mb = total_params * 4 / (1024 ** 2)
    print(f"\nBackbone parameters:      {backbone_params:,}")
    print(f"Head parameters:          {head_params:,}")
    print(f"Total parameters:         {total_params:,}")
    print(f"MACs per inference:       {macs:,} at {MODEL_RES}x{MODEL_RES} input "
          f"(approx.; conv + dense layers; {macs_skipped} layers skipped)")
    print(f"Float32 weight size:      {float32_size_mb:.2f} MB")
    timer, best = EpochTimer(), BestWeights()
    backbone.trainable = False
    compile_model(model, args.head_lr)
    head_trainable = count_trainable(model)
    print(f"\n{'=' * 70}\nPhase 1: head only (backbone frozen), "
          f"{head_trainable:,} trainable parameters, lr={args.head_lr}\n{'=' * 70}")
    t0 = time.perf_counter()
    head_history = model.fit(train_ds, validation_data=val_ds,
                             epochs=args.head_epochs,
                             callbacks=[lr_callback(), best, timer], verbose=2)
    set_fine_tune(backbone, args.fine_tune_layers)
    compile_model(model, args.fine_lr)
    fine_trainable = count_trainable(model)
    print(f"\n{'=' * 70}\nPhase 2: fine-tuning top {args.fine_tune_layers} backbone layers, "
          f"{fine_trainable:,} trainable parameters, lr={args.fine_lr}\n{'=' * 70}")
    fine_history = model.fit(train_ds, validation_data=val_ds,
                             initial_epoch=args.head_epochs, epochs=TOTAL_EPOCHS,
                             callbacks=[lr_callback(), best, timer], verbose=2)
    total_time = time.perf_counter() - t0
    
    if best.weights is not None:
        model.set_weights(best.weights)
    print(f"\nRestored best weights: val loss {best.best:.4f} @ epoch {best.epoch}")
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    logits = model.predict(test_ds, verbose=0)
    main_preds = logits.argmax(axis=1)
    
    history = {
        key: [float(v) for v in head_history.history.get(key, [])]
        + [float(v) for v in fine_history.history.get(key, [])]
        for key in set(head_history.history) | set(fine_history.history)
    }
    n_epochs = len(history["val_loss"])
    train_loss, val_loss = history["loss"], history["val_loss"]
    val_acc, train_acc = history["val_accuracy"], history["accuracy"]
    et = timer.epoch_times
    fine_et = et[args.head_epochs:]
    
    model_path = RESULTS_DIR / "model.keras"
    model.save(model_path)
    disk_size_kb = model_path.stat().st_size / 1024
    disk_size_mb = disk_size_kb / 1024
    print(f"\nSaved model file size: {disk_size_mb:.2f} MB ({disk_size_kb:.1f} KB)")
    
    report_dict = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, output_dict=True, zero_division=0)
    report_text = classification_report(
        y_test, main_preds, labels=list(range(num_classes)),
        target_names=class_names, zero_division=0)
    print("\n" + report_text)
    cm = confusion_matrix(y_test, main_preds, labels=list(range(num_classes)))
    
    final_metrics = {
        "command": COMMAND_LINE,
        "model": MODEL_LABEL,
        "weights": args.weights,
        "backbone_input_resolution": MODEL_RES,
        "augmentation": AUG_MODE,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "test_precision_macro": float(precision_score(y_test, main_preds, average="macro", zero_division=0)),
        "test_recall_macro": float(recall_score(y_test, main_preds, average="macro", zero_division=0)),
        "test_f1_macro": float(f1_score(y_test, main_preds, average="macro", zero_division=0)),
        "test_f1_weighted": float(f1_score(y_test, main_preds, average="weighted", zero_division=0)),
        "test_classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
        "total_parameters": total_params,
        "backbone_parameters": backbone_params,
        "head_parameters": head_params,
        "trainable_parameters_phase1": head_trainable,
        "trainable_parameters_phase2": fine_trainable,
        "macs_per_inference": macs,
        "float32_weight_size_mb": float(float32_size_mb),
        "model_file_size_kb": float(disk_size_kb),
        "model_file_size_mb": float(disk_size_mb),
        "epochs_completed": n_epochs,
        "best_validation_loss": float(best.best),
        "best_validation_loss_epoch": int(best.epoch),
        "best_validation_accuracy": float(max(val_acc)),
        "best_validation_accuracy_epoch": int(np.argmax(val_acc)) + 1,
        "final_training_loss": float(train_loss[-1]),
        "final_training_accuracy": float(train_acc[-1]),
        "final_validation_loss": float(val_loss[-1]),
        "final_validation_accuracy": float(val_acc[-1]),
        "total_training_time_seconds": float(total_time),
        "average_epoch_time_seconds": float(np.mean(et)),
        "average_epoch_time_excl_first_seconds": float(np.mean(et[1:])) if len(et) > 1 else float(np.mean(et)),
        "median_epoch_time_seconds": float(np.median(et)),
        "median_epoch_time_fine_tune_seconds": float(np.median(fine_et)) if fine_et else None,
    }
    (RESULTS_DIR / "metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
    (RESULTS_DIR / "training_history.json").write_text(
        json.dumps({**history, "epoch_time_seconds": et}, indent=2), encoding="utf-8")
    (RESULTS_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
    np.savetxt(RESULTS_DIR / "confusion_matrix.csv", cm, delimiter=",", fmt="%d")
    
    (RESULTS_DIR / "run_metadata.json").write_text(json.dumps({
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "command": COMMAND_LINE,
        "arguments": vars(args),
        "model": MODEL_LABEL,
        "dataset": str(DATASET_DIR),
        "data_pipeline_image_size": list(IMG_SIZE),
        "backbone_input_resolution": MODEL_RES,
        "batch_size": BATCH_SIZE,
        "head_epochs": args.head_epochs,
        "fine_tune_epochs": args.fine_epochs,
        "fine_tune_layers": args.fine_tune_layers,
        "optimizer": f"Adam (head lr={args.head_lr}, fine-tune lr={args.fine_lr})",
        "loss": "SparseCategoricalCrossentropy(from_logits=True)",
        "lr_schedule": "ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7) per phase",
        "seed": SEED,
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "num_images": len(paths),
        "train_images": len(X_train),
        "validation_images": len(X_val),
        "test_images": len(X_test),
    }, indent=2), encoding="utf-8")
    
    print(f"\n{MODEL_LABEL}: test acc {test_acc:.4f} | macro F1 "
          f"{final_metrics['test_f1_macro']:.4f} | params {total_params:,} | "
          f"{disk_size_mb:.2f} MB | {final_metrics['median_epoch_time_seconds']:.2f}s/epoch (median)")
    plt.figure(figsize=(12, 6))
    plt.bar(class_names, [class_counts[class_to_idx[n]] for n in class_names])
    plt.title("Images per Class")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
    plt.close()
    
    epochs_axis = range(1, n_epochs + 1)
    boundary = args.head_epochs + 0.5
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, train_loss, label="Training Loss")
    plt.plot(epochs_axis, val_loss, label="Validation Loss")
    plt.axvline(boundary, color="gray", linestyle="--", label="unfreeze backbone")
    plt.title(f"{MODEL_LABEL}: Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=150)
    plt.close()
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_axis, train_acc, label="Training Accuracy")
    plt.plot(epochs_axis, val_acc, label="Validation Accuracy")
    plt.axvline(boundary, color="gray", linestyle="--", label="unfreeze backbone")
    plt.title(f"{MODEL_LABEL}: Training and Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "accuracy_curve.png", dpi=150)
    plt.close()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
    ax.set_title(f"{MODEL_LABEL}: Test Confusion Matrix")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()
    
    print(f"\nResults saved to: {RESULTS_DIR}")


## Interpretation of pretrained-model results

Compare the pretrained networks with Model B using the current experiment results. Discuss whether any accuracy improvement justifies the additional parameters, model size, MACs, and training time for the intended edge device. Do not assume that either pretrained network is best before the cells have completed.


## 6. Final comparison and trade-off discussion

The comparison below combines the completed notebook runs. Model B is the constrained custom model required for edge deployment; MobileNetV2 and EfficientNet-B0 show the accuracy and resource trade-offs available from pretrained lightweight architectures.

The final conclusion should be based on the current run and should identify the best-performing model, the smallest model, and the most suitable deployment option separately. These may not be the same model.


## Comparison of completed experiments

In [ ]:
rows = []
for name, result in NOTEBOOK_RESULTS.items():
    rows.append((name, result['test_accuracy'], result['test_f1_macro'], result['parameters']))
    print(f"{name}: accuracy={result['test_accuracy']:.4f}, macro F1={result['test_f1_macro']:.4f}, parameters={result['parameters']:,}")
if rows:
    best_accuracy = max(rows, key=lambda row: row[1])
    smallest_model = min(rows, key=lambda row: row[3])
    print(f"\nBest test accuracy in this run: {best_accuracy[0]} ({best_accuracy[1]:.4f})")
    print(f"Smallest model in this run: {smallest_model[0]} ({smallest_model[3]:,} parameters)")


In [ ]:
if False:
    """Compare every saved modelA and modelB optimizer run."""
    
    import argparse
    import csv
    import json
    from pathlib import Path
    
    
    METRICS = [
        ("Total parameters", "total_parameters", ",.0f"),
        ("Trainable parameters", "trainable_parameters", ",.0f"),
        ("Estimated model size on disk (KB)", "model_file_size_kb", ",.1f"),
        ("Float32 weight size (KB)", "float32_weight_size_kb", ",.1f"),
        ("MACs per inference", "macs_per_inference", ",.0f"),
        ("Training time per epoch (s)", "average_epoch_time_seconds", ".2f"),
        ("Median epoch time (s)", "median_epoch_time_seconds", ".2f"),
        ("Test loss", "test_loss", ".4f"),
        ("Test accuracy", "test_accuracy", ".4f"),
        ("Test precision (macro)", "test_precision_macro", ".4f"),
        ("Test recall (macro)", "test_recall_macro", ".4f"),
        ("Test F1 (macro)", "test_f1_macro", ".4f"),
        ("Test F1 (weighted)", "test_f1_weighted", ".4f"),
        ("Best validation loss", "best_validation_loss", ".4f"),
        ("Best validation accuracy", "best_validation_accuracy", ".4f"),
    ]
    
    
    def load_results(results_dir):
        """Load each saved run from results/modelA/<optimizer> and modelB/<optimizer>."""
        loaded = {}
        for model_name in ("modelA", "modelB"):
            model_dir = results_dir / model_name
            if not model_dir.is_dir():
                continue
            for metrics_path in sorted(model_dir.glob("*/metrics.json")):
                try:
                    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError) as error:
                    print(f"Skipping {metrics_path}: {error}")
                    continue
                optimizer = metrics_path.parent.name
                run_name = f"{model_name}_{optimizer}"
                loaded[run_name] = {
                    "model": model_name,
                    "optimizer_key": optimizer,
                    "optimizer": metrics.get("optimizer", optimizer),
                    "metrics_path": str(metrics_path),
                    **metrics,
                }
        return loaded
    
    
    def format_value(value, format_spec):
        if value is None:
            return "N/A"
        return format(value, format_spec)
    
    
    def build_markdown(results):
        lines = [
            "# Saved Model Comparison",
            "",
            "Each column is one saved model and optimizer run. Missing values are shown as `N/A`.",
            "",
            "| Metric | " + " | ".join(results) + " |",
            "|---|" + "---:|" * len(results),
        ]
    
        for label, key, format_spec in METRICS:
            values = [
                format_value(results[run_name].get(key), format_spec)
                for run_name in results
            ]
            lines.append(f"| {label} | " + " | ".join(values) + " |")
    
        if not results:
            return "# Saved Model Comparison\n\nNo optimizer results were found.\n"
        return "\n".join(lines).rstrip() + "\n"
    
    
    def build_csv_rows(results):
        rows = []
        for run_name, run in results.items():
            row = {
                "run": run_name,
                "model": run["model"],
                "optimizer_key": run["optimizer_key"],
                "optimizer": run.get("optimizer", run["optimizer_key"]),
                "metrics_path": run["metrics_path"],
            }
            row.update({key: run.get(key) for _, key, _ in METRICS})
            rows.append(row)
        return rows
    
    
    def main():
        parser = argparse.ArgumentParser(
            description="Compare every saved modelA and modelB optimizer run."
        )
        parser.add_argument(
            "--results-dir",
            type=Path,
            default=Path(__file__).resolve().parent / "results",
            help="directory containing modelA and modelB result folders",
        )
        parser.add_argument(
            "--output-dir",
            type=Path,
            help="directory for generated comparison files; defaults to results directory",
        )
        args = parser.parse_args()
    
        results_dir = args.results_dir.resolve()
        output_dir = (args.output_dir or results_dir).resolve()
        output_dir.mkdir(parents=True, exist_ok=True)
        results = load_results(results_dir)
    
        markdown_path = output_dir / "comparison_table_AB.md"
        csv_path = output_dir / "comparison_table_AB.csv"
        json_path = output_dir / "comparison_table_AB.json"
    
        markdown_path.write_text(build_markdown(results), encoding="utf-8")
        rows = build_csv_rows(results)
        fieldnames = ["run", "model", "optimizer_key", "optimizer", "metrics_path"] + [
            key for _, key, _ in METRICS
        ]
        with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    
        json_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"Comparison report saved to: {markdown_path}")
        print(f"CSV data saved to: {csv_path}")
        print(f"JSON data saved to: {json_path}")
    
    
    if __name__ == "__main__":
        main()


In [ ]:
if False:
    """Compare saved modelB, MobileNetV2, and EfficientNetB0 results."""
    
    import argparse
    import csv
    import json
    from pathlib import Path
    
    
    MODEL_DIRS = {
        "modelB": ("modelB",),
        "MobileNetV2": (("mobilenetv2",), ("mobile", "mobilenetv2")),
        "EfficientNetB0": (("efficientnetb0",), ("mobile", "efficientnetb0")),
    }
    METRICS = [
        ("Total parameters", "total_parameters", ",.0f"),
        ("Trainable parameters", "trainable_parameters", ",.0f"),
        ("Model size (MB)", "model_size_mb", ".2f"),
        ("Model size (KB)", "model_file_size_kb", ",.1f"),
        ("Float32 weight size (MB)", "float32_weight_size_mb", ".2f"),
        ("MACs per inference", "macs_per_inference", ",.0f"),
        ("Average epoch time (s)", "average_epoch_time_seconds", ".2f"),
        ("Median epoch time (s)", "median_epoch_time_seconds", ".2f"),
        ("Test loss", "test_loss", ".4f"),
        ("Test accuracy", "test_accuracy", ".4f"),
        ("Test precision (macro)", "test_precision_macro", ".4f"),
        ("Test recall (macro)", "test_recall_macro", ".4f"),
        ("Test F1 (macro)", "test_f1_macro", ".4f"),
        ("Test F1 (weighted)", "test_f1_weighted", ".4f"),
    ]
    
    
    def load_metrics(results_dir):
        runs = {}
        for label, locations in MODEL_DIRS.items():
            if label == "modelB":
                model_dir = results_dir / locations[0]
                paths = sorted(model_dir.glob("*/metrics.json"))
            else:
                paths = [
                    results_dir.joinpath(*location) / "metrics.json"
                    for location in locations
                ]
            for metrics_path in paths:
                if not metrics_path.is_file():
                    continue
                try:
                    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError) as error:
                    print(f"Skipping {metrics_path}: {error}")
                    continue
                run_name = (
                    f"modelB_{metrics_path.parent.name}"
                    if label == "modelB"
                    else label
                )
                runs[run_name] = {
                    "model": label,
                    "metrics_path": str(metrics_path),
                    **metrics,
                }
        return runs
    
    
    def metric_value(metrics, key):
        value = metrics.get(key)
        if value is None and key == "trainable_parameters":
            value = metrics.get("trainable_parameters_phase2")
        if value is None and key == "model_size_mb":
            size_kb = metrics.get("model_file_size_kb")
            if size_kb is not None:
                return size_kb / 1024
        if value is None and key == "model_file_size_kb":
            size_mb = metrics.get("model_size_mb")
            if size_mb is not None:
                return size_mb * 1024
        if value is None and key == "float32_weight_size_mb":
            size_kb = metrics.get("float32_weight_size_kb")
            if size_kb is not None:
                return size_kb / 1024
        return value
    
    
    def format_value(value, format_spec):
        return "N/A" if value is None else format(value, format_spec)
    
    
    def build_markdown(runs):
        if not runs:
            return "# modelB and SOTA Comparison\n\nNo saved results found.\n"
        lines = [
            "# modelB and Lightweight SOTA Comparison",
            "",
            "Each column is a saved run. Missing models or metrics are shown as `N/A`.",
            "",
            "| Metric | " + " | ".join(runs) + " |",
            "|---|" + "---:|" * len(runs),
        ]
        for label, key, format_spec in METRICS:
            values = [
                format_value(metric_value(runs[run], key), format_spec)
                for run in runs
            ]
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        return "\n".join(lines) + "\n"
    
    
    def build_csv_rows(runs):
        rows = []
        for run_name, metrics in runs.items():
            row = {
                "run": run_name,
                "model": metrics["model"],
                "metrics_path": metrics["metrics_path"],
            }
            for _, key, _ in METRICS:
                row[key] = metric_value(metrics, key)
            rows.append(row)
        return rows
    
    
    def main():
        parser = argparse.ArgumentParser(
            description="Compare saved resource-constrained modelB with MobileNetV2 and EfficientNetB0."
        )
        parser.add_argument(
            "--results-dir",
            type=Path,
            default=Path(__file__).resolve().parent / "results",
        )
        parser.add_argument("--output-dir", type=Path)
        args = parser.parse_args()
    
        results_dir = args.results_dir.resolve()
        output_dir = (args.output_dir or results_dir).resolve()
        output_dir.mkdir(parents=True, exist_ok=True)
        runs = load_metrics(results_dir)
        markdown_path = output_dir / "comparison_modelB_sota.md"
        csv_path = output_dir / "comparison_modelB_sota.csv"
        json_path = output_dir / "comparison_modelB_sota.json"
        markdown_path.write_text(build_markdown(runs), encoding="utf-8")
        fieldnames = ["run", "model", "metrics_path"] + [key for _, key, _ in METRICS]
        with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(build_csv_rows(runs))
        json_path.write_text(json.dumps(runs, indent=2), encoding="utf-8")
        print(f"Comparison report saved to: {markdown_path}")
        print(f"CSV data saved to: {csv_path}")
        print(f"JSON data saved to: {json_path}")
    
    
    if __name__ == "__main__":
        main()


## References

1. UCI Machine Learning Repository, [RealWaste dataset](https://archive.ics.uci.edu/dataset/908/realwaste).
2. Howard et al., [MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications](https://arxiv.org/abs/1704.04861).
3. Sandler et al., [MobileNetV2: Inverted Residuals and Linear Bottlenecks](https://arxiv.org/abs/1801.04381).
4. Tan and Le, [EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks](https://arxiv.org/abs/1905.11946).
